# Statcast Pitch Mix Analysis

This notebook demonstrates how `polars-baseball` fetches Statcast pitch-level data
and immediately works with native Polars expressions — no pandas conversion needed.

**Prerequisites:**
```bash
pip install polars-baseball
pip install "polars-baseball[plot]"    # optional, for charts
```

In [ ]:
from __future__ import annotations

import polars as pl

import polars_baseball as pb

## 1. Fetch one day of Statcast data

Single-day fetch runs in a few seconds. The result is a `polars.DataFrame`.

> **Finding player IDs:** Use `pb.playerid_lookup("Cole", "Gerrit")` or
> `pb.player_name_suggestions("cole")` to look up the `key_mlbam` for any player.

In [ ]:
df = await pb.statcast(start_date="2026-06-01", end_date="2026-06-01")
print(f"Rows: {df.height}, Columns: {len(df.columns)}")
df.head(3)

## 2. Pitcher arsenal breakdown

Group by `pitch_type` to show how often each pitch is thrown and its average velocity.

In [ ]:
pitch_mix = (
    df.group_by("pitch_type")
    .agg(
        pl.len().alias("pitch_count"),
        pl.col("release_speed").mean().round(1).alias("avg_velocity"),
    )
    .with_columns((pl.col("pitch_count") / pl.col("pitch_count").sum() * 100).round(1).alias("usage_pct"))
    .sort("pitch_count", descending=True)
)
pitch_mix

## 3. Top 5 hardest-throwing pitchers

Aggregate by pitcher name to find max fastball velocity.

In [ ]:
hardest_fastballs = (
    df.filter(pl.col("pitch_type").is_in(["FF", "FA", "SI", "FC"]))
    .group_by("player_name")
    .agg(pl.col("release_speed").max().alias("max_velo"))
    .sort("max_velo", descending=True)
    .head(5)
)
hardest_fastballs

## 4. Batted-ball outcomes by pitch type

See which pitch types produce which events (singles, home runs, etc.).

In [ ]:
outcomes = (
    df.filter(pl.col("events").is_not_null()).group_by(["pitch_type", "events"]).len().sort("len", descending=True)
)
outcomes.head(10)

## 5. Strike-zone visualization (optional)

Requires `polars-baseball[plot]` (installs hvPlot + Altair).

If the plot extra is not installed, this cell is harmless — just skip it.

In [ ]:
try:
    import hvplot.polars  # noqa: F401

    chart = (
        df.filter(pl.col("plate_x").is_not_null() & pl.col("plate_z").is_not_null())
        .to_pandas()
        .plot.scatter(x="plate_x", y="plate_z", c="pitch_type", alpha=0.6)
    )
    print("Scatter plot rendered. Close the window to continue.")
except ImportError:
    print("hvPlot not installed. Run: pip install 'polars-baseball[plot]'")

## Summary

- `polars-baseball` returns native `polars.DataFrame` — zero conversion overhead.
- All aggregation, filtering, and reshaping is done with Polars expressions.
- Async fetching means you can compose multiple data sources in parallel.

Next steps:
- Try `pb.fangraphs.batting(start_season=2026, end_season=2026, qual=100)`
- Try `pb.mlb.schedule(date="2026-06-01")`
- See [notebooks/fangraphs_leaderboard_demo.ipynb](fangraphs_leaderboard_demo.ipynb)